In [ ]:
# 1. Setup

from pathlib import Path
import re
import pandas as pd
from IPython.display import display, HTML

# Project directories
PROJECT_ROOT = Path.cwd().parent

raw_data_dir = PROJECT_ROOT / "data"
transcript_dir = raw_data_dir / "Justice_GBV_VS_Interviews"
datalist_path = raw_data_dir / "DataList.xlsx"

output_dir = PROJECT_ROOT / "outputs"
processed_dir = PROJECT_ROOT / "processed"

output_dir.mkdir(exist_ok=True)
processed_dir.mkdir(exist_ok=True)

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

# Jupyter dataframe style
display(HTML("""
<style>
.dataframe {
    font-size: 13px;
    border-collapse: collapse;
}

.dataframe th {
    text-align: left !important;
    white-space: nowrap;
    padding: 6px 8px;
}

.dataframe td {
    text-align: left !important;
    white-space: nowrap;
    padding: 6px 8px;
}

.dataframe tbody tr:nth-child(even) {
    background-color: #f7f7f7;
}
</style>
"""))

def show_table(df, cols=None, n=10):
    """
    Display a cleaner preview of a dataframe.
    """
    if cols is not None:
        cols = [col for col in cols if col in df.columns]
        df = df[cols]
    
    display(
        df.head(n).style.set_properties(**{
            "text-align": "left",
            "white-space": "nowrap"
        }).set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("white-space", "nowrap")
                ]
            }
        ])
    )

# Check paths
print("Base directory:", base_dir)
print("Raw data directory exists:", raw_data_dir.exists())
print("Transcript directory exists:", transcript_dir.exists())
print("DataList file exists:", datalist_path.exists())
print("Output directory exists:", output_dir.exists())
print("Processed directory exists:", processed_dir.exists())

In [ ]:
# 2. Load DataList.xlsx

datalist = pd.read_excel(
    datalist_path,
    sheet_name="Participant overview",
    header=1
)

# Clean column names
datalist.columns = (
    datalist.columns
    .astype(str)
    .str.strip()
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)

# Remove fully empty rows
datalist = datalist.dropna(how="all").copy()

# Keep rows that have participant ID
datalist = datalist[datalist["Initial Participant ID"].notna()].copy()

# Standardise participant ID
datalist["participant_id"] = (
    datalist["Initial Participant ID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("DataList shape:", datalist.shape)
print("Number of participant records:", datalist["participant_id"].nunique())

In [ ]:
# 3. Build transcript file inventory

def extract_just_ids_from_filename(file_stem):
    """
    Extract participant IDs from transcript filenames.

    Handles both:
    - JUST001
    - JUST174-177

    Returns a list of participant IDs.
    """
    file_stem = str(file_stem).upper().strip()

    match = re.match(r"JUST(\d+)(?:-(\d+))?", file_stem)

    if not match:
        return []

    start_num = int(match.group(1))
    end_num = int(match.group(2)) if match.group(2) else start_num
    width = len(match.group(1))

    return [
        f"JUST{num:0{width}d}"
        for num in range(start_num, end_num + 1)
    ]


# List all Word transcript files
word_files = sorted([
    f for f in transcript_dir.rglob("*")
    if f.is_file()
    and f.suffix.lower() in [".doc", ".docx"]
    and not f.name.startswith("~$")
])

file_inventory = pd.DataFrame({
    "file_name": [f.name for f in word_files],
    "file_stem": [f.stem for f in word_files],
    "file_suffix": [f.suffix.lower() for f in word_files],
    "file_path": [str(f) for f in word_files],
    "file_size_bytes": [f.stat().st_size for f in word_files],
})

file_inventory["participant_ids"] = file_inventory["file_stem"].apply(
    extract_just_ids_from_filename
)

file_inventory["participant_ids_str"] = file_inventory["participant_ids"].apply(
    lambda ids: ";".join(ids)
)

file_inventory["n_participant_ids"] = file_inventory["participant_ids"].apply(len)

file_inventory["shared_transcript"] = file_inventory["n_participant_ids"] > 1

print("Number of transcript files:", len(file_inventory))

print("\nFile suffix counts:")
print(file_inventory["file_suffix"].value_counts())

print("\nShared transcript files:")
show_table(
    file_inventory[file_inventory["shared_transcript"] == True],
    cols=[
        "file_name",
        "participant_ids_str",
        "n_participant_ids"
    ],
    n=10
)


print("\nFiles with no participant ID extracted:")
show_table(
    file_inventory[file_inventory["n_participant_ids"] == 0],
    cols=[
        "file_name",
        "file_stem",
        "file_suffix"
    ],
    n=10
)


print("\nFile inventory preview:")

show_table(
    file_inventory,
    cols=[
        "file_name",
        "file_stem",
        "file_suffix",
        "file_size_bytes",
        "participant_ids_str",
        "n_participant_ids",
        "shared_transcript"
    ],
    n=10
)

In [ ]:
# 4.1 Match DataList participant records with transcript file inventory

file_inventory_exploded = (
    file_inventory
    .explode("participant_ids")
    .rename(columns={"participant_ids": "participant_id"})
    .copy()
)

file_inventory_exploded["participant_id"] = (
    file_inventory_exploded["participant_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

datalist_matching = datalist.merge(
    file_inventory_exploded[
        [
            "participant_id",
            "file_name",
            "file_path",
            "file_suffix",
            "file_size_bytes",
            "shared_transcript",
            "participant_ids_str"
        ]
    ],
    on="participant_id",
    how="left"
)

datalist_matching["has_transcript_file"] = datalist_matching["file_name"].notna()

print("DataList participant records:", len(datalist))
print("DataList records after matching:", len(datalist_matching))

print("\n")
print(datalist_matching["has_transcript_file"].value_counts(dropna=False))

print("\n")
print(datalist_matching["shared_transcript"].value_counts(dropna=False))

print("\nDataList-transcript matching preview:")
show_table(
    datalist_matching,
    cols=[
        "participant_id",
        "file_name",
        "has_transcript_file",
        "shared_transcript",
        "participant_ids_str"
    ],
    n=15
)

DataList contains 152 participant-level records.
The transcript folder contains 145 Word files.
After expanding shared transcript files, 150 file-participant links were identified.
Of the 152 DataList records, 149 could be matched to a transcript file, while 3 had no matched transcript file.
Seven DataList records are linked to shared transcript files.

In [ ]:
# 4.2 Inspect DataList records without matched transcript files

unmatched_datalist_records = datalist_matching[
    datalist_matching["has_transcript_file"] == False
].copy()

print(
    "Number of DataList records without matched transcript file:",
    len(unmatched_datalist_records)
)

display_cols = [
    "Initial Participant ID",
    "participant_id",
    "Individual or group interview",
    "Age",
    "Identified Gender",
    "Identified Ethnicity",
    "BME?",
    "Nationality"
]

display_cols = [
    col for col in display_cols
    if col in unmatched_datalist_records.columns
]

print()
show_table(
    unmatched_datalist_records,
    cols=display_cols,
    n=10
)

Three DataList records, JUST041, JUST279 and JUST312, did not have corresponding transcript files in the available transcript folder. These records are retained in the DataList-transcript matching output, but excluded from transcript-based text analysis.

In [ ]:
# 4.3 Check transcript participant IDs against DataList participant IDs

datalist_ids = set(datalist["participant_id"])
transcript_ids = set(file_inventory_exploded["participant_id"])

transcript_ids_not_in_datalist = sorted(transcript_ids - datalist_ids)
datalist_ids_not_in_transcripts = sorted(datalist_ids - transcript_ids)

print("Participant IDs in transcript files but not in DataList:")
print(transcript_ids_not_in_datalist)

print()
print("Participant IDs in DataList but not in transcript files:")
print(datalist_ids_not_in_transcripts)

One transcript file ID, JUST196, was present in the transcript folder but not represented in the Datalist file.

In [ ]:
# 4.4 Inspect transcript participant ID not found in DataList

extra_transcript_files = file_inventory_exploded[
    file_inventory_exploded["participant_id"].isin(transcript_ids_not_in_datalist)
].copy()

print("Transcript participant IDs not found in DataList:")
print(transcript_ids_not_in_datalist)

print()
show_table(
    extra_transcript_files,
    cols=[
        "participant_id",
        "file_name",
        "file_suffix",
        "file_size_bytes",
        "shared_transcript",
        "participant_ids_str"
    ],
    n=10
)


# Inspect DataList participant IDs around JUST190-JUST200

datalist_id_numbers = (
    datalist["participant_id"]
    .str.extract(r"JUST(\d+)", expand=False)
    .astype(int)
)

datalist_190_200 = datalist[
    datalist_id_numbers.between(190, 200)
].copy()

print()
print("DataList participant IDs around JUST190-JUST200:")

show_table(
    datalist_190_200,
    cols=[
        "Initial Participant ID",
        "participant_id"
    ],
    n=20
)

### Matching summary

DataList.xlsx contains 152 participant-level records. The transcript folder contains 145 Word files. Two transcript files are shared transcripts: JUST174-177 and JUST296-298. After expanding shared transcript files, 150 file-participant links were identified.

Of the 152 DataList records, 149 were matched to transcript files. Three DataList records did not have matched transcript files: JUST041, JUST279 and JUST312.

One transcript participant ID, JUST196, was present in the transcript folder but not found in DataList. This mismatch is recorded at the file-inventory stage. Its inclusion in the final analysis corpus will be decided after text extraction and corpus construction.

In [ ]:
# 5.1 Extract transcript text from Word files

import time

try:
    import win32com.client as win32
except ImportError:
    raise ImportError(
        "win32com is not available. Please install pywin32 first, e.g. pip install pywin32"
    )


def clean_transcript_text(text):
    """
    Lightly clean transcript text while preserving readability.
    This does not remove stopwords or alter substantive content.
    """
    if text is None:
        return ""

    text = str(text)

    # Normalise common Word control characters and spacing
    text = text.replace("\xa0", " ")
    text = text.replace("\x0b", "\n")
    text = text.replace("\r", "\n")

    # Remove excessive spaces but keep paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def count_words(text):
    """
    Count word-like tokens for basic transcript length statistics.
    """
    if not isinstance(text, str) or text.strip() == "":
        return 0

    return len(re.findall(r"\b\w+\b", text))


def read_word_document(file_path, word_app):
    """
    Read text from a Word document using Microsoft Word.
    Works for both .doc and .docx files if Word can open them.
    """
    doc = None

    try:
        doc = word_app.Documents.Open(
            FileName=str(file_path),
            ReadOnly=True,
            AddToRecentFiles=False,
            Visible=False
        )

        text_raw = doc.Content.Text
        text_clean = clean_transcript_text(text_raw)

        return {
            "read_status": "success",
            "read_error": "",
            "text_raw": text_raw,
            "text_clean": text_clean,
            "char_count": len(text_clean),
            "word_count": count_words(text_clean)
        }

    except Exception as e:
        return {
            "read_status": "failed",
            "read_error": f"{type(e).__name__}: {e}",
            "text_raw": "",
            "text_clean": "",
            "char_count": 0,
            "word_count": 0
        }

    finally:
        if doc is not None:
            doc.Close(False)


transcript_records = []

word_app = win32.DispatchEx("Word.Application")
word_app.Visible = False
word_app.DisplayAlerts = 0

start_time = time.time()

try:
    for i, row in file_inventory.iterrows():
        if (i + 1) % 25 == 0 or i == 0:
            print(f"Reading file {i + 1}/{len(file_inventory)}: {row['file_name']}")

        result = read_word_document(row["file_path"], word_app)

        transcript_records.append({
            "file_name": row["file_name"],
            "file_stem": row["file_stem"],
            "file_suffix": row["file_suffix"],
            "file_path": row["file_path"],
            "file_size_bytes": row["file_size_bytes"],
            "participant_ids_str": row["participant_ids_str"],
            "n_participant_ids": row["n_participant_ids"],
            "shared_transcript": row["shared_transcript"],
            **result
        })

finally:
    word_app.Quit()

end_time = time.time()

transcript_texts = pd.DataFrame(transcript_records)

print()
print("Text extraction completed.")
print("Time used:", round(end_time - start_time, 2), "seconds")
print("Transcript files attempted:", len(transcript_texts))

print()
print("Read status counts:")
print(transcript_texts["read_status"].value_counts(dropna=False))

In [ ]:
# 5.2 Check failed transcript reads

failed_transcripts = transcript_texts[
    transcript_texts["read_status"] != "success"
].copy()

print("Number of failed transcript reads:", len(failed_transcripts))

if len(failed_transcripts) > 0:
    show_table(
        failed_transcripts,
        cols=[
            "file_name",
            "file_suffix",
            "file_size_bytes",
            "participant_ids_str",
            "read_status",
            "read_error"
        ],
        n=20
    )
else:
    print("All transcript files were successfully read.")

In [ ]:
# 5.3 Preview extracted transcript text statistics

print("Transcript text statistics:")
print()

print(transcript_texts[["char_count", "word_count"]].describe())

print()
print("Shortest transcripts:")

show_table(
    transcript_texts.sort_values("word_count", ascending=True),
    cols=[
        "file_name",
        "participant_ids_str",
        "read_status",
        "word_count",
        "char_count",
        "shared_transcript"
    ],
    n=10
)

print()
print("Longest transcripts:")

show_table(
    transcript_texts.sort_values("word_count", ascending=False),
    cols=[
        "file_name",
        "participant_ids_str",
        "read_status",
        "word_count",
        "char_count",
        "shared_transcript"
    ],
    n=10
)

All 145 transcript files were successfully read. Transcript lengths range from 624 to 23,099 words, with a mean of approximately 10,321 words and a median of approximately 10,061 words. No empty or unreadable transcripts were detected. One transcript file is explicitly marked as partial in the filename, and two files are shared transcripts covering multiple participant IDs.

In [ ]:
# Inspect the shortest transcript text preview

shortest_transcript = transcript_texts.sort_values("word_count", ascending=True).iloc[0]

print("Shortest transcript file:", shortest_transcript["file_name"])
print("Word count:", shortest_transcript["word_count"])
print()
print(shortest_transcript["text_clean"][:1500])

In [ ]:
# Check transcripts marked as partial in filenames

partial_transcripts = transcript_texts[
    transcript_texts["file_name"].str.contains("partial", case=False, na=False)
].copy()

print("Number of transcript files marked as partial:", len(partial_transcripts))

show_table(
    partial_transcripts,
    cols=[
        "file_name",
        "participant_ids_str",
        "word_count",
        "char_count",
        "read_status"
    ],
    n=10
)

In [ ]:
# 5.4 Add transcript quality flags

transcript_texts["partial_transcript"] = transcript_texts["file_name"].str.contains(
    "partial",
    case=False,
    na=False
)

transcript_texts["interview_notes_no_audio"] = transcript_texts["text_clean"].str.contains(
    "Interview notes",
    case=False,
    na=False
) & transcript_texts["text_clean"].str.contains(
    "no audio",
    case=False,
    na=False
)

transcript_texts["non_standard_text_type"] = (
    transcript_texts["partial_transcript"]
    | transcript_texts["interview_notes_no_audio"]
)

print("Partial transcript files:")
print(transcript_texts["partial_transcript"].value_counts(dropna=False))

print()
print("Interview notes / no audio files:")
print(transcript_texts["interview_notes_no_audio"].value_counts(dropna=False))

print()
print("Non-standard text type files:")
print(transcript_texts["non_standard_text_type"].value_counts(dropna=False))

print()
show_table(
    transcript_texts[transcript_texts["non_standard_text_type"] == True],
    cols=[
        "file_name",
        "participant_ids_str",
        "word_count",
        "char_count",
        "partial_transcript",
        "interview_notes_no_audio",
        "read_status"
    ],
    n=20
)

### Transcript extraction and quality check summary

All 145 transcript files were successfully read. Transcript lengths range from 624 to 23,099 words, with a mean of approximately 10,321 words and a median of approximately 10,061 words. No empty or unreadable files were detected.

Two files require additional quality flags. JUST153 is explicitly marked as a partial transcript in the filename. JUST247 appears to be interview notes rather than a full audio-based transcript, as the text begins with “Interview notes – no audio”. These files are retained at the text extraction stage but flagged as non-standard text types. Their inclusion in the final analysis corpus will be decided during corpus construction.

In [ ]:
# 5.5 Save extracted transcript texts and length statistics

cleaned_transcripts_dir = processed_dir / "cleaned_transcripts"
cleaned_transcripts_dir.mkdir(exist_ok=True)

cleaned_text_paths = []

for _, row in transcript_texts.iterrows():
    cleaned_text_path = cleaned_transcripts_dir / f"{row['file_stem']}.txt"

    if row["read_status"] == "success":
        cleaned_text_path.write_text(row["text_clean"], encoding="utf-8")
        cleaned_text_paths.append(str(cleaned_text_path))
    else:
        cleaned_text_paths.append("")

transcript_texts["cleaned_text_path"] = cleaned_text_paths

transcript_texts_path = output_dir / "transcript_texts_v2.csv"
transcript_length_stats_path = output_dir / "transcript_length_stats_v2.csv"

transcript_texts.to_csv(
    transcript_texts_path,
    index=False,
    encoding="utf-8-sig"
)

transcript_texts[
    [
        "file_name",
        "file_stem",
        "file_suffix",
        "participant_ids_str",
        "n_participant_ids",
        "shared_transcript",
        "read_status",
        "read_error",
        "char_count",
        "word_count",
        "partial_transcript",
        "interview_notes_no_audio",
        "non_standard_text_type",
        "cleaned_text_path"
    ]
].to_csv(
    transcript_length_stats_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved transcript texts to:", transcript_texts_path)
print("Saved transcript length statistics to:", transcript_length_stats_path)
print("Saved cleaned transcript txt files to:", cleaned_transcripts_dir)

In [ ]:
# 6.1 Build file-level analysis corpus

def split_participant_ids(ids_str):
    """
    Split semicolon-separated participant IDs into a list.
    """
    if pd.isna(ids_str) or str(ids_str).strip() == "":
        return []
    return [x.strip().upper() for x in str(ids_str).split(";") if x.strip() != ""]


datalist_ids = set(datalist["participant_id"])

analysis_corpus = transcript_texts.copy()

analysis_corpus["analysis_level"] = "file_level"

analysis_corpus["participant_id_list"] = analysis_corpus["participant_ids_str"].apply(
    split_participant_ids
)

analysis_corpus["linked_datalist_ids"] = analysis_corpus["participant_id_list"].apply(
    lambda ids: [pid for pid in ids if pid in datalist_ids]
)

analysis_corpus["transcript_only_ids"] = analysis_corpus["participant_id_list"].apply(
    lambda ids: [pid for pid in ids if pid not in datalist_ids]
)

analysis_corpus["linked_datalist_ids_str"] = analysis_corpus["linked_datalist_ids"].apply(
    lambda ids: ";".join(ids)
)

analysis_corpus["transcript_only_ids_str"] = analysis_corpus["transcript_only_ids"].apply(
    lambda ids: ";".join(ids)
)

analysis_corpus["n_linked_datalist_records"] = analysis_corpus["linked_datalist_ids"].apply(len)

analysis_corpus["has_linked_datalist_record"] = (
    analysis_corpus["n_linked_datalist_records"] > 0
)

analysis_corpus["transcript_without_datalist_record"] = (
    analysis_corpus["transcript_only_ids"].apply(len) > 0
)

analysis_corpus["text_lower"] = analysis_corpus["text_clean"].str.lower()

analysis_corpus["include_in_kwic_corpus_preliminary"] = (
    analysis_corpus["read_status"] == "success"
)

analysis_corpus["requires_review_before_final_corpus"] = (
    analysis_corpus["non_standard_text_type"]
    | analysis_corpus["transcript_without_datalist_record"]
)

print("File-level transcript records:", len(analysis_corpus))
print()

print("Preliminary KWIC corpus inclusion:")
print(analysis_corpus["include_in_kwic_corpus_preliminary"].value_counts(dropna=False))

print()
print("Has linked DataList record:")
print(analysis_corpus["has_linked_datalist_record"].value_counts(dropna=False))

print()
print("Requires review before final corpus:")
print(analysis_corpus["requires_review_before_final_corpus"].value_counts(dropna=False))

print()
show_table(
    analysis_corpus,
    cols=[
        "file_name",
        "participant_ids_str",
        "linked_datalist_ids_str",
        "transcript_only_ids_str",
        "shared_transcript",
        "partial_transcript",
        "interview_notes_no_audio",
        "non_standard_text_type",
        "has_linked_datalist_record",
        "transcript_without_datalist_record",
        "include_in_kwic_corpus_preliminary",
        "requires_review_before_final_corpus",
        "word_count"
    ],
    n=15
)

In [ ]:
# Inspect records requiring review before final corpus

review_cases = analysis_corpus[
    analysis_corpus["requires_review_before_final_corpus"] == True
].copy()

print("Number of records requiring review before final corpus:", len(review_cases))

show_table(
    review_cases,
    cols=[
        "file_name",
        "participant_ids_str",
        "linked_datalist_ids_str",
        "transcript_only_ids_str",
        "partial_transcript",
        "interview_notes_no_audio",
        "transcript_without_datalist_record",
        "word_count"
    ],
    n=10
)

In [ ]:
# 6.2 Add selected DataList variables to file-level corpus

datalist_lookup = datalist.set_index("participant_id", drop=False)

datalist_variable_map = {
    "Initial Participant ID": "datalist_initial_participant_id",
    "Individual or group interview": "datalist_interview_type",
    "Age": "datalist_age",
    "Identified Gender": "datalist_gender",
    "Identified Ethnicity": "datalist_ethnicity",
    "BME?": "datalist_bme",
    "Nationality": "datalist_nationality"
}

datalist_variable_map = {
    source_col: new_col
    for source_col, new_col in datalist_variable_map.items()
    if source_col in datalist.columns
}


def get_single_datalist_value(linked_ids, source_col):
    """
    Return a DataList value only when the transcript is linked to exactly one DataList record.
    For shared transcripts or transcript-only files, return blank.
    """
    if len(linked_ids) != 1:
        return ""

    pid = linked_ids[0]

    if pid not in datalist_lookup.index:
        return ""

    value = datalist_lookup.loc[pid, source_col]

    if isinstance(value, pd.Series):
        value = value.iloc[0]

    if pd.isna(value):
        return ""

    return str(value).strip()


for source_col, new_col in datalist_variable_map.items():
    analysis_corpus[new_col] = analysis_corpus["linked_datalist_ids"].apply(
        lambda ids: get_single_datalist_value(ids, source_col)
    )


analysis_corpus["datalist_variable_assignment"] = "single_linked_datalist_record"

analysis_corpus.loc[
    analysis_corpus["shared_transcript"] == True,
    "datalist_variable_assignment"
] = "shared_transcript_multiple_datalist_records"

analysis_corpus.loc[
    analysis_corpus["has_linked_datalist_record"] == False,
    "datalist_variable_assignment"
] = "no_linked_datalist_record"


print("DataList variable assignment:")
print(analysis_corpus["datalist_variable_assignment"].value_counts(dropna=False))

print()
show_table(
    analysis_corpus,
    cols=[
        "file_name",
        "participant_ids_str",
        "linked_datalist_ids_str",
        "shared_transcript",
        "datalist_variable_assignment",
        "datalist_age",
        "datalist_gender",
        "datalist_ethnicity",
        "datalist_bme",
        "datalist_nationality",
        "word_count"
    ],
    n=20
)

### DataList variables in the file-level corpus

The main analysis corpus is constructed at the transcript-file level, with one row representing one transcript file. This avoids duplicating shared transcript files during keyword matching, KWIC extraction and later annotation.

DataList variables are attached directly only when a transcript file is linked to a single DataList participant record. For shared transcript files, DataList variables are not collapsed into a single combined value, because this could incorrectly treat multiple participants as one case. Instead, shared transcript files are flagged as `shared_transcript_multiple_datalist_records`.

Transcript files without a linked DataList record are flagged separately. They may be inspected as text files if needed, but they are not used in any DataList-based exploratory comparison because no participant-level DataList variables are available for them.

In [ ]:
# 6.3 Add corpus decision notes

def build_corpus_note(row):
    notes = []

    if row["read_status"] != "success":
        notes.append("read_failed")

    if row["shared_transcript"]:
        notes.append("shared_transcript")

    if row["partial_transcript"]:
        notes.append("partial_transcript")

    if row["interview_notes_no_audio"]:
        notes.append("interview_notes_no_audio")

    if row["transcript_without_datalist_record"]:
        notes.append("transcript_id_not_found_in_datalist")

    if len(notes) == 0:
        return "standard_transcript"

    return "; ".join(notes)


analysis_corpus["corpus_note"] = analysis_corpus.apply(
    build_corpus_note,
    axis=1
)

print("Corpus note counts:")
print(analysis_corpus["corpus_note"].value_counts(dropna=False))

print()
show_table(
    analysis_corpus[analysis_corpus["corpus_note"] != "standard_transcript"],
    cols=[
        "file_name",
        "participant_ids_str",
        "linked_datalist_ids_str",
        "transcript_only_ids_str",
        "corpus_note",
        "datalist_variable_assignment",
        "word_count",
        "has_linked_datalist_record",
        "include_in_kwic_corpus_preliminary",
        "requires_review_before_final_corpus"
    ],
    n=20
)

### Corpus notes

A `corpus_note` field was added to document transcript-level issues relevant to later analysis. Standard transcript files are marked as `standard_transcript`. Shared transcript files, partial transcripts, interview notes without audio, and transcript files without a linked DataList record are flagged separately.

These notes do not automatically exclude files from the preliminary KWIC corpus. They provide an audit trail for later decisions about final corpus inclusion, annotation sampling and DataList-based exploratory comparison.

In [ ]:
# 6.4 Build DataList-text link table

transcript_text_summary_cols = [
    "file_name",
    "read_status",
    "read_error",
    "char_count",
    "word_count",
    "partial_transcript",
    "interview_notes_no_audio",
    "non_standard_text_type",
    "cleaned_text_path"
]

transcript_text_summary_cols = [
    col for col in transcript_text_summary_cols
    if col in transcript_texts.columns
]

datalist_text_links = datalist_matching.merge(
    transcript_texts[transcript_text_summary_cols],
    on="file_name",
    how="left"
)

datalist_text_links["has_readable_text"] = (
    datalist_text_links["read_status"] == "success"
)

print("DataList-text link records:", len(datalist_text_links))

print()
print(datalist_text_links["has_transcript_file"].value_counts(dropna=False))

print()
print(datalist_text_links["has_readable_text"].value_counts(dropna=False))

print()
datalist_text_links_preview = datalist_text_links.copy()

datalist_text_links_preview["word_count"] = (
    datalist_text_links_preview["word_count"]
    .fillna(0)
    .astype(int)
)

show_table(
    datalist_text_links_preview,
    cols=[
        "participant_id",
        "file_name",
        "has_transcript_file",
        "has_readable_text",
        "shared_transcript",
        "word_count",
        "partial_transcript",
        "interview_notes_no_audio"
    ],
    n=15
)

### DataList-text link table

A DataList-text link table was constructed at the participant-record level. This table retains all 152 DataList records and records whether each participant record has a matched transcript file and readable extracted text.

Of the 152 DataList records, 149 are linked to readable transcript text. Three DataList records, JUST041, JUST279 and JUST312, have no matched transcript file and therefore no readable transcript text. This table is used for documenting DataList-transcript matching and for later DataList-based exploratory comparison, rather than for file-level KWIC analysis.

In [ ]:
# 6.5 Save analysis corpus outputs

analysis_corpus_path = output_dir / "analysis_corpus_file_level_v2.csv"
datalist_text_links_path = output_dir / "datalist_text_links_v2.csv"

analysis_corpus.to_csv(
    analysis_corpus_path,
    index=False,
    encoding="utf-8-sig"
)

datalist_text_links.to_csv(
    datalist_text_links_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved file-level analysis corpus to:", analysis_corpus_path)
print("Saved DataList-text link table to:", datalist_text_links_path)

In [ ]:
# 6.6 Corpus construction summary

summary_lines = [
    "Corpus construction summary",
    "---------------------------",
    f"DataList participant-level records: {len(datalist)}",
    f"Transcript Word files: {len(file_inventory)}",
    f"Expanded file-participant links: {len(file_inventory_exploded)}",
    f"File-level analysis corpus records: {len(analysis_corpus)}",
    f"Successfully read transcript files: {(analysis_corpus['read_status'] == 'success').sum()}",
    f"DataList records matched to transcript files: {datalist_matching['has_transcript_file'].sum()}",
    f"DataList records without matched transcript files: {(~datalist_matching['has_transcript_file']).sum()}",
    f"Transcript files with no linked DataList record: {analysis_corpus['transcript_without_datalist_record'].sum()}",
    f"Shared transcript files: {analysis_corpus['shared_transcript'].sum()}",
    f"Partial transcript files: {analysis_corpus['partial_transcript'].sum()}",
    f"Interview notes / no audio files: {analysis_corpus['interview_notes_no_audio'].sum()}",
    f"Non-standard text type files: {analysis_corpus['non_standard_text_type'].sum()}",
    f"Preliminary KWIC corpus records: {analysis_corpus['include_in_kwic_corpus_preliminary'].sum()}",
]

summary_text = "\n".join(summary_lines)

print(summary_text)

summary_path = output_dir / "corpus_construction_summary_v2.txt"

summary_path.write_text(summary_text, encoding="utf-8")

print()
print("Saved corpus construction summary to:", summary_path)

## Data preparation completed

This notebook prepared the transcript corpus for later keyword-assisted analysis. DataList.xlsx contains 152 participant-level records, while the transcript folder contains 145 Word files. All 145 transcript files were successfully read and converted into cleaned text files.

The main analysis corpus was constructed at the transcript-file level, with one row per transcript file, to avoid duplicating shared transcripts. The file-level analysis corpus contains 145 records. A separate DataList-text link table was also created to document the relationship between the 152 DataList records and available transcript files.

Several data boundary issues were documented: three DataList records have no matched transcript files, one transcript participant ID is not found in DataList, two files are shared transcripts, one file is marked as a partial transcript, and one file appears to be interview notes without audio. These cases are flagged for later methodological decisions.